In [1]:
!pip install mysql.connector.python
!pip install pandas
!pip install numpy

In [2]:
import mysql.connector
import pandas as pd
import numpy as np

myconnect = mysql.connector.connect(
    user = "root",
    password = "",
    port = "3307",
    host = "localhost"
)

mycursor = myconnect.cursor(buffered = True)



In [3]:
mycursor.execute("CREATE DATABASE ubereats_restaurant")

In [4]:
mycursor.execute("USE ubereats_restaurant")

In [5]:
mycursor.execute("""
CREATE TABLE IF NOT EXISTS restaurants_data (
    id INT AUTO_INCREMENT PRIMARY KEY,
    restaurant_name VARCHAR(255),
    online_order BOOLEAN,
    table_booking BOOLEAN,
    rating_outof_5 FLOAT,
    votes INT,
    restaurant_phone VARCHAR(50),
    location VARCHAR(255),
    restaurant_type VARCHAR(100),
    dish_liked TEXT,
    cuisines TEXT,
    approx_cost_for_2 FLOAT,
    service_type VARCHAR(100),
    city_hub VARCHAR(100),
    rating_category VARCHAR(50),
    pricing_segment VARCHAR(50)
)
""")

In [7]:
data = pd.read_csv('C:\\Users\\welcome\\Desktop\\guvi\\uber_eats project\\cleaned_ubereatsdata_version1.csv')

In [8]:
data.head()

,restaurant_name,online_order,table_booking,rating_outof_5,votes,restaurant_phone,location,restaurant_type,dish_liked,cuisines,approx_cost_for_2,service_type,city_hub,rating_category,pricing_segment
0,jalsa,True,True,4.1,775,080 42297555 +91 9743772233,banashankari,casual dining,"pasta, lunch buffet, masala papad, paneer laja...","north indian, mughlai, chinese",800.0,buffet,banashankari,Excellent,Mid-Range
1,spice elephant,True,False,4.1,787,080 41714161,banashankari,casual dining,"momos, lunch buffet, chocolate nirvana, thai g...","chinese, north indian, thai",800.0,buffet,banashankari,Excellent,Mid-Range
2,san churro cafe,True,False,3.8,918,+91 9663487993,banashankari,"cafe, casual dining","churros, cannelloni, minestrone soup, hot choc...","cafe, mexican, italian",800.0,buffet,banashankari,Good,Mid-Range
3,addhuri udupi bhojana,False,False,3.7,88,+91 9620009302,banashankari,quick bites,masala dosa,"south indian, north indian",300.0,buffet,banashankari,Good,Budget
4,grand village,False,False,3.8,166,+91 8026612447 +91 9901210005,basavanagudi,casual dining,"panipuri, gol gappe","north indian, rajasthani",600.0,buffet,banashankari,Good,Mid-Range


In [9]:
data_to_insert = [tuple(x) for x in data[['restaurant_name', 'online_order', 'table_booking', 'rating_outof_5',
       'votes', 'restaurant_phone', 'location', 'restaurant_type',
       'dish_liked', 'cuisines', 'approx_cost_for_2', 'service_type',
       'city_hub', 'rating_category', 'pricing_segment']].values]

insert_query = """
INSERT INTO restaurants_data (
    restaurant_name, online_order, table_booking, rating_outof_5,
       votes, restaurant_phone, location, restaurant_type,
       dish_liked, cuisines, approx_cost_for_2, service_type,
       city_hub, rating_category, pricing_segment
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""


batch_size = 1000
for i in range(0, len(data_to_insert), batch_size):
    batch = data_to_insert[i:i+batch_size]
    mycursor.executemany(insert_query, batch)
    myconnect.commit() 
    print(f"{i+len(batch)} rows uploaded...")


1000 rows uploaded...
2000 rows uploaded...
3000 rows uploaded...
4000 rows uploaded...
5000 rows uploaded...
6000 rows uploaded...
7000 rows uploaded...
8000 rows uploaded...
9000 rows uploaded...
10000 rows uploaded...
11000 rows uploaded...
12000 rows uploaded...
13000 rows uploaded...
14000 rows uploaded...
15000 rows uploaded...
16000 rows uploaded...
17000 rows uploaded...
18000 rows uploaded...
19000 rows uploaded...
20000 rows uploaded...
21000 rows uploaded...
22000 rows uploaded...
23000 rows uploaded...
23012 rows uploaded...


In [5]:
data.shape

(23012, 15)

In [16]:
mycursor.execute("SHOW TABLES;")
for table in mycursor.fetchall():
    print(table)

('restaurants_data',)


In [39]:
# 1. Which Bangalore locations have the highest average restaurant ratings?
# Business Value: Identifies premium-performing areas suitable for brand positioning and new partner onboarding

# hint : "I used a threshold of 25 to ensure statistical significance and filter out locations with very few data points"

from tabulate import tabulate

mycursor.execute("""SELECT location, ROUND(AVG(rating_outof_5),1) as avg_rating, COUNT(*) as restaurant_count
FROM restaurants_data
GROUP BY location
HAVING restaurant_count > 25
ORDER BY avg_rating DESC
LIMIT 10""")

out = mycursor.fetchall()
print(tabulate(out, headers = [i[0] for i in mycursor.description], tablefmt='psql'))



+-----------------------+--------------+--------------------+
| location              |   avg_rating |   restaurant_count |
|-----------------------+--------------+--------------------|
| koramangala 5th block |          4.2 |               1759 |
| lavelle road          |          4.2 |                442 |
| koramangala 3rd block |          4.1 |                161 |
| residency road        |          4.1 |                440 |
| st. marks road        |          4.1 |                304 |
| cunningham road       |          4.1 |                333 |
| koramangala 2nd block |          4.1 |                 45 |
| sadashiv nagar        |          4.1 |                 38 |
| koramangala 8th block |          4   |                122 |
| ulsoor                |          4   |                472 |
+-----------------------+--------------+--------------------+


In [13]:
mycursor.execute("SHOW TABLES;")
for table in mycursor.fetchall():
    print(table)

('restaurants_data',)


In [17]:
# 2. Which locations are over-saturated with restaurants?
# Business Value: Helps avoid overcrowded markets and guides smarter expansion decisions.

mycursor.execute("""SELECT location, COUNT(*) AS restaurant_count
FROM restaurants_data
GROUP BY location
ORDER BY restaurant_count DESC
LIMIT 10""")

out = mycursor.fetchall()
print(tabulate(out, headers = [i[0] for i in mycursor.description], tablefmt='psql'))


+-----------------------+--------------------+
| location              |   restaurant_count |
|-----------------------+--------------------|
| koramangala 5th block |               1759 |
| btm                   |               1444 |
| indiranagar           |               1331 |
| hsr                   |               1154 |
| jayanagar             |               1030 |
| jp nagar              |                989 |
| whitefield            |                821 |
| koramangala 6th block |                718 |
| koramangala 7th block |                715 |
| marathahalli          |                678 |
+-----------------------+--------------------+


In [28]:
# 3. Does online ordering improve restaurant ratings?
# Business Value: Evaluates the ROI of Uber Eats’ online ordering feature for partners.

mycursor.execute("""SELECT online_order, AVG(rating_outof_5) AS avg_rating, COUNT(*) AS restaurant_count
FROM restaurants_data
GROUP BY online_order""")

out = mycursor.fetchall()
print(tabulate(out, headers = [i[0] for i in mycursor.description], tablefmt='psql')) 

+----------------+--------------+--------------------+
|   online_order |   avg_rating |   restaurant_count |
|----------------+--------------+--------------------|
|              0 |      3.92955 |               6738 |
|              1 |      3.89343 |              16274 |
+----------------+--------------+--------------------+


In [38]:
# 4.Does table booking correlate with higher customer ratings?
# Business Value: Measures the effectiveness of table booking as a premium feature.

mycursor.execute("""SELECT pricing_segment,table_booking,
    COUNT(*) AS total_restaurants,
    ROUND(AVG(rating_outof_5),1) AS avg_rating
FROM restaurants_data
GROUP BY pricing_segment, table_booking
ORDER BY pricing_segment, avg_rating ASC""")

out = mycursor.fetchall()
print(tabulate(out, headers = [i[0] for i in mycursor.description], tablefmt='psql'))

+-------------------+-----------------+---------------------+--------------+
| pricing_segment   |   table_booking |   total_restaurants |   avg_rating |
|-------------------+-----------------+---------------------+--------------|
| Budget            |               0 |                9638 |          3.8 |
| Budget            |               1 |                 112 |          3.9 |
| Mid-Range         |               0 |                6267 |          3.8 |
| Mid-Range         |               1 |                2318 |          4.1 |
| Premium           |               0 |                1086 |          4   |
| Premium           |               1 |                3591 |          4.2 |
+-------------------+-----------------+---------------------+--------------+


In [47]:
# 5. What price range delivers the best customer satisfaction?
# Business Value: Helps define the optimal pricing segment for partner success.

mycursor.execute("""
    SELECT pricing_segment, COUNT(*) AS total_restaurants, ROUND(AVG(rating_outof_5), 2) AS avg_rating,
    ROUND(COUNT(CASE WHEN rating_category = 'Excellent' THEN 1 END) * 100.0 / COUNT(*), 2) AS excellent_percentage
    FROM restaurants_data
    GROUP BY pricing_segment
    ORDER BY avg_rating DESC
""")

out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))



+-------------------+---------------------+--------------+------------------------+
| pricing_segment   |   total_restaurants |   avg_rating |   excellent_percentage |
|-------------------+---------------------+--------------+------------------------|
| Premium           |                4677 |         4.18 |                  81.23 |
| Mid-Range         |                8585 |         3.89 |                  50.16 |
| Budget            |                9750 |         3.79 |                  37.11 |
+-------------------+---------------------+--------------+------------------------+


In [61]:
# 6. How do low, mid, and premium-priced restaurants perform in terms of ratings?
# Business Value: Supports pricing-based market segmentation strategies

mycursor.execute("""SELECT pricing_segment, ROUND(AVG(rating_outof_5),1) AS avg_rating, COUNT(*) AS restaurant_count,
SUM(votes) AS total_customer_engagement FROM restaurants_data
GROUP BY pricing_segment
ORDER BY avg_rating DESC
""")

out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))


+-------------------+--------------+--------------------+-----------------------------+
| pricing_segment   |   avg_rating |   restaurant_count |   total_customer_engagement |
|-------------------+--------------+--------------------+-----------------------------|
| Premium           |          4.2 |               4677 |                 6.20395e+06 |
| Mid-Range         |          3.9 |               8585 |                 5.41531e+06 |
| Budget            |          3.8 |               9750 |                 2.28226e+06 |
+-------------------+--------------+--------------------+-----------------------------+


In [77]:
# 7. Which cuisines are most common in Bangalore?
# Business Value: Reveals market demand and cuisine saturation levels.

from collections import Counter

mycursor.execute("SELECT cuisines FROM restaurants_data WHERE cuisines IS NOT NULL")
rows = mycursor.fetchall()

all_cuisines = []
for row in rows:
    split_names = [i.strip() for i in row[0].split(',')]
    all_cuisines.extend(split_names)

cuisine_counts = Counter(all_cuisines).most_common(10)

print(tabulate(cuisine_counts, headers=['Cuisine Name', 'Frequency (Count)'], tablefmt='psql'))


+----------------+---------------------+
| Cuisine Name   |   Frequency (Count) |
|----------------+---------------------|
| north indian   |                9907 |
| chinese        |                7383 |
| continental    |                4120 |
| cafe           |                3509 |
| fast food      |                3237 |
| south indian   |                2787 |
| italian        |                2699 |
| desserts       |                2609 |
| biryani        |                2536 |
| beverages      |                2296 |
+----------------+---------------------+


In [ ]:
# 8. Which cuisines receive the highest average ratings?
# Business Value: Identifies high-quality cuisine categories suitable for promotion.

In [ ]:
# 9. Which cuisines perform well despite having fewer restaurants?
# Business Value: Highlights niche opportunities for differentiation.

In [100]:
# 10.What is the relationship between restaurant cost and rating?
# Business Value: Determines whether higher pricing translates to better customer perception



############doubt#############

mycursor.execute("""SELECT rating_category, 
AVG(approx_cost_for_2) AS avg_cost, 
MIN(approx_cost_for_2) AS min_cost, 
MAX(approx_cost_for_2) AS max_cost,
COUNT(*) AS restaurant_count FROM restaurants_data 
GROUP BY rating_category
ORDER BY rating_category DESC
""")

out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+-------------------+------------+------------+------------+--------------------+
| rating_category   |   avg_cost |   min_cost |   max_cost |   restaurant_count |
|-------------------+------------+------------+------------+--------------------|
| Good              |    593.342 |         40 |       3000 |              10116 |
| Excellent         |    912.093 |        100 |       6000 |              11723 |
| Average           |    553.725 |        150 |       1500 |               1173 |
+-------------------+------------+------------+------------+--------------------+


In [99]:
# 11. Which locations are ideal for premium restaurant onboarding?
# Business Value: Combines cost, rating, and location insights to guide premium expansion.

mycursor.execute("""
SELECT location, AVG(approx_cost_for_2) AS avg_cost, AVG(rating_outof_5) AS avg_rating, COUNT(*) AS restaurant_count
FROM restaurants_data
GROUP BY location
HAVING avg_cost > (SELECT AVG(approx_cost_for_2) FROM restaurants_data) AND avg_rating > 3.5
ORDER BY avg_cost DESC, avg_rating DESC
""")

out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+-----------------------+------------+--------------+--------------------+
| location              |   avg_cost |   avg_rating |   restaurant_count |
|-----------------------+------------+--------------+--------------------|
| sankey road           |   2802.94  |      4.10588 |                 17 |
| mg road               |   1473.09  |      3.95025 |                589 |
| lavelle road          |   1426.58  |      4.19253 |                442 |
| race course road      |   1284     |      3.98667 |                 75 |
| infantry road         |   1186.17  |      3.8883  |                 94 |
| richmond road         |   1124.79  |      4.02906 |                351 |
| residency road        |   1091.14  |      4.05136 |                440 |
| ulsoor                |   1070.97  |      4.02119 |                472 |
| langford town         |   1000     |      3.9     |                 22 |
| old airport road      |    996.683 |      3.81188 |                202 |
| cunningham road       |

In [107]:
#12. Which locations show high demand but lower average ratings?
#Business Value: Indicates areas where quality improvement initiatives are needed.

mycursor.execute("""SELECT location, COUNT(*) AS total_restaurants, ROUND(AVG(rating_outof_5),2) AS avg_rating
FROM restaurants_data
GROUP BY location
HAVING COUNT(*) > 50 
   AND AVG(rating_outof_5) < 3.7
ORDER BY total_restaurants DESC, avg_rating ASC
""")

out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))


+--------------------+---------------------+--------------+
| location           |   total_restaurants |   avg_rating |
|--------------------+---------------------+--------------|
| bannerghatta road  |                 496 |         3.68 |
| bellandur          |                 459 |         3.68 |
| electronic city    |                 320 |         3.7  |
| brookefield        |                 307 |         3.7  |
| domlur             |                 206 |         3.69 |
| banaswadi          |                 195 |         3.64 |
| jeevan bhima nagar |                 174 |         3.68 |
| shanti nagar       |                 166 |         3.66 |
| commercial street  |                 141 |         3.69 |
| nagawara           |                  62 |         3.67 |
| kumaraswamy layout |                  56 |         3.62 |
+--------------------+---------------------+--------------+


In [118]:
# 13.Do restaurants offering both online ordering and table booking perform better?
# Business Value: Validates bundled feature adoption for partners

mycursor.execute("""SELECT online_order, table_booking, ROUND(AVG(rating_outof_5),1) AS avg_rating, AVG(votes) AS avg_popularity,
COUNT(*) AS restaurant_count FROM restaurants_data
GROUP BY online_order, table_booking
ORDER BY avg_rating DESC
""")

out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))



+----------------+-----------------+--------------+------------------+--------------------+
|   online_order |   table_booking |   avg_rating |   avg_popularity |   restaurant_count |
|----------------+-----------------+--------------+------------------+--------------------|
|              0 |               1 |          4.2 |          1223.39 |               2407 |
|              1 |               1 |          4.1 |          1209.63 |               3614 |
|              1 |               0 |          3.8 |           357.23 |              12660 |
|              0 |               0 |          3.8 |           476.26 |               4331 |
+----------------+-----------------+--------------+------------------+--------------------+


In [ ]:
#14.What combination of factors maximizes restaurant success on Uber Eats?
#(Pricing + Location + Cuisine + Platform Features)
#Business Value: Supports strategic partner recommendations.

In [122]:
#15.Which restaurants are top performers within each pricing segment?
#Business Value: Helps identify benchmark partners and best practices

mycursor.execute("""SELECT pricing_segment, restaurant_name, rating_outof_5, location
FROM restaurants_data
WHERE (pricing_segment, rating_outof_5) IN (SELECT pricing_segment, MAX(rating_outof_5) FROM restaurants_data
GROUP BY pricing_segment)
ORDER BY pricing_segment
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))


+-------------------+--------------------------------+------------------+-----------------------+
| pricing_segment   | restaurant_name                |   rating_outof_5 | location              |
|-------------------+--------------------------------+------------------+-----------------------|
| Budget            | belgian waffle factory         |              4.9 | brigade road          |
| Budget            | belgian waffle factory         |              4.9 | koramangala 5th block |
| Budget            | belgian waffle factory         |              4.9 | brigade road          |
| Budget            | milano ice cream               |              4.9 | indiranagar           |
| Budget            | belgian waffle factory         |              4.9 | brigade road          |
| Budget            | belgian waffle factory         |              4.9 | brigade road          |
| Budget            | belgian waffle factory         |              4.9 | brigade road          |
| Budget            

In [124]:
mycursor.execute(""" WITH RankedRestaurants AS (SELECT restaurant_name, location, approx_cost_for_2, rating_outof_5,
pricing_segment,
DENSE_RANK() OVER (PARTITION BY pricing_segment ORDER BY rating_outof_5 DESC, votes DESC) AS rnk
FROM restaurants_data)
SELECT * FROM RankedRestaurants 
WHERE rnk <= 5 
ORDER BY pricing_segment, rnk
""")

out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+-----------------------------+-----------------------+---------------------+------------------+-------------------+-------+
| restaurant_name             | location              |   approx_cost_for_2 |   rating_outof_5 | pricing_segment   |   rnk |
|-----------------------------+-----------------------+---------------------+------------------+-------------------+-------|
| milano ice cream            | indiranagar           |                 400 |              4.9 | Budget            |     1 |
| belgian waffle factory      | brigade road          |                 400 |              4.9 | Budget            |     2 |
| belgian waffle factory      | brigade road          |                 400 |              4.9 | Budget            |     2 |
| belgian waffle factory      | brigade road          |                 400 |              4.9 | Budget            |     3 |
| belgian waffle factory      | brigade road          |                 400 |              4.9 | Budget            |     3 |


In [129]:
mycursor.execute("""
WITH GroupedRestaurants AS (
    SELECT restaurant_name, pricing_segment, MAX(rating_outof_5) AS max_rating, MAX(votes) AS max_votes, MAX(location) AS location
    FROM restaurants_data
    GROUP BY restaurant_name, pricing_segment),
RankedRestaurants AS (SELECT restaurant_name, location, max_rating AS rating, pricing_segment,
DENSE_RANK() OVER (PARTITION BY pricing_segment ORDER BY max_rating DESC, max_votes DESC) AS rank
FROM GroupedRestaurants)
SELECT * FROM RankedRestaurants 
WHERE rank <= 5
ORDER BY pricing_segment, rank
""")

out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+--------------------------------+-----------------------+----------+-------------------+--------+
| restaurant_name                | location              |   rating | pricing_segment   |   rank |
|--------------------------------+-----------------------+----------+-------------------+--------|
| milano ice cream               | jayanagar             |      4.9 | Budget            |      1 |
| belgian waffle factory         | vasanth nagar         |      4.9 | Budget            |      2 |
| ctr                            | malleshwaram          |      4.8 | Budget            |      3 |
| brahmin's coffee bar           | malleshwaram          |      4.8 | Budget            |      4 |
| o.g. variar & sons             | rajajinagar           |      4.8 | Budget            |      5 |
| santãâãâãâãâãâãâãâãâãâãâãâãâãâãâãâãâ© spa cuisine                                | indiranagar           |      4.9 | Mid-Range         |      1 |
| house of commons          

In [130]:
data.columns

Index(['restaurant_name', 'online_order', 'table_booking', 'rating_outof_5',
       'votes', 'restaurant_phone', 'location', 'restaurant_type',
       'dish_liked', 'cuisines', 'approx_cost_for_2', 'service_type',
       'city_hub', 'rating_category', 'pricing_segment'],
      dtype='object')

In [131]:
mycursor.execute("""
SELECT location, ROUND(AVG(rating_outof_5),2) AS avg_rating
FROM restaurants_data
GROUP BY location
""")

out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+-------------------------------+--------------+
| location                      |   avg_rating |
|-------------------------------+--------------|
| banashankari                  |         3.81 |
| banaswadi                     |         3.64 |
| bannerghatta road             |         3.68 |
| basavanagudi                  |         3.84 |
| basaveshwara nagar            |         3.82 |
| bellandur                     |         3.68 |
| bommanahalli                  |         3.14 |
| brigade road                  |         3.94 |
| brookefield                   |         3.7  |
| btm                           |         3.76 |
| central bangalore             |         3.6  |
| church street                 |         4.05 |
| city market                   |         3.93 |
| commercial street             |         3.69 |
| cunningham road               |         4.1  |
| cv raman nagar                |         3.63 |
| domlur                        |         3.69 |
| east bangalore    